## Pruebas Algoritmos

In [15]:
import pandas as pd
import numpy as np
import time
import os
import json
from tqdm import tqdm 
from typing import Tuple

# ✅ PASO 1: Importa el archivo con tus funciones de detección
import algoritmos_deteccion as ad

# =============================================================================
# SECCIÓN DE EVALUACIÓN
# =============================================================================

def calcular_iou(real: Tuple[int, int], detectado: Tuple[int, int]) -> float:
    """Calcula el Índice Jaccard (Intersection over Union) entre dos intervalos."""
    inicio_real, fin_real = real
    inicio_det, fin_det = detectado
    interseccion = max(0, min(fin_real, fin_det) - max(inicio_real, inicio_det))
    union = (fin_real - inicio_real) + (fin_det - inicio_det) - interseccion
    return interseccion / union if union > 0 else 0.0

def encontrar_mejor_coincidencia(real: Tuple[int, int], detectados: list) -> tuple:
    """Encuentra el intervalo detectado con el mayor IoU para un evento real."""
    if not detectados:
        return None, 0.0
    ious = [calcular_iou(real, det) for det in detectados]
    mejor_iou = max(ious)
    mejor_intervalo = detectados[np.argmax(ious)]
    return mejor_intervalo, mejor_iou

# --- CONFIGURACIÓN PRINCIPAL ---

BASE_DIR = r'..\Generacion_Señales'
DIRS = {
    'Sag': os.path.join(BASE_DIR, 'Sag'),
    'Sag_Armonic': os.path.join(BASE_DIR, 'Sag_Armonic'),
    'Sag_Multiple': os.path.join(BASE_DIR, 'Sag_Multiple'),
    'Sag_Notch': os.path.join(BASE_DIR, 'Sag_Notch'),
    'Sag_Transitorios': os.path.join(BASE_DIR, 'Sag_Transitorios')
}

print("Cargando y consolidando metadatos...")
todos_los_metadata = []
for dir_nombre, dir_path in DIRS.items():
    for filename in os.listdir(dir_path):
        if 'metadata' in filename:
            try:
                df_meta = pd.read_csv(os.path.join(dir_path, filename))
                df_meta['SourceDir'] = dir_nombre
                todos_los_metadata.append(df_meta)
                print(f"  - Cargado: {filename} ({len(df_meta)} filas)")
            except Exception as e:
                print(f"Error cargando {filename}: {e}")

metadata_df = pd.concat(todos_los_metadata, ignore_index=True)
print(f"\nTotal de señales a evaluar: {len(metadata_df)}")

mapa_archivos = {
    'sags.csv': ['sag'],
    'sags_con_ruido.csv': ['sag_ruido'],
    'sags_con_armonicos.csv': ['armonicos_solo'],
    'sags_con_armonicos_y_ruido.csv': ['armonicos_y_ruido'],
    'sags_multiples.csv': ['multiple_limpio'],
    'sags_multiples_con_ruido.csv': ['multiple_con_ruido'],
    'sags_con_muescas.csv': ['muescas_limpio'],
    'sags_con_muescas_y_ruido.csv': ['muescas_con_ruido'],
    'sags_con_transitorios.csv': ['transitorio_limpio'],
    'sags_con_transitorios_y_ruido.csv': ['transitorio_con_ruido']
}

tipo_a_archivo = {tipo: (DIRS[dirname], fname) for dirname, fnames in {
    'Sag': ['sags.csv', 'sags_con_ruido.csv'],
    'Sag_Armonic': ['sags_con_armonicos.csv', 'sags_con_armonicos_y_ruido.csv'],
    'Sag_Multiple': ['sags_multiples.csv', 'sags_multiples_con_ruido.csv'],
    'Sag_Notch': ['sags_con_muescas.csv', 'sags_con_muescas_y_ruido.csv'],
    'Sag_Transitorios': ['sags_con_transitorios.csv', 'sags_con_transitorios_y_ruido.csv']
}.items() for fname in fnames for tipo in mapa_archivos.get(fname, [])}

# --- EJECUCIÓN DE LA EVALUACIÓN (VERSIÓN OPTIMIZADA) ---

fs = 7200
algoritmos = [
    ("RMS", lambda senal: ad.detectar_eventos_rms(senal, fs=fs)),
    ("Hilbert", lambda senal: ad.detectar_eventos_hilbert(senal, fs=fs)),
    ("SWT", lambda senal: ad.detectar_eventos_swt(senal, fs=fs)),
    ("Fusion", lambda senal: ad.detectar_eventos_con_fusion(senal, fs=fs))
]

# ✅ **OPTIMIZACIÓN: Crear un caché para los DataFrames cargados**
signal_data_cache = {}
resultados = []

print("\nIniciando evaluación con caché de datos optimizado...")
for _, fila in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Evaluando señales"):
    
    id_sag = fila['ID_Sag']
    
    # ✅ --- INICIA LA CORRECCIÓN ---
    # Lógica mejorada para manejar metadatos antiguos sin la columna 'tipo'
    tipo_evento = fila['tipo']
    if pd.isna(tipo_evento):
        # Si el tipo es NaN, es de los metadatos antiguos. Lo deducimos del ID.
        if 'ruido' in id_sag.lower():
            tipo_evento = 'sag_ruido'
        else:
            tipo_evento = 'sag'
    # ✅ --- TERMINA LA CORRECCIÓN ---

    try:
        # Construye la ruta completa al archivo CSV necesario
        dir_path, archivo_csv_nombre = tipo_a_archivo[tipo_evento]
        ruta_completa_csv = os.path.join(dir_path, archivo_csv_nombre)

        # 1. Revisa si el archivo ya está en el caché
        if ruta_completa_csv not in signal_data_cache:
            # Si no está, cárgalo UNA VEZ y guárdalo en el caché
            signal_data_cache[ruta_completa_csv] = pd.read_csv(ruta_completa_csv, index_col='Tiempo')
        
        # 2. Obtén la señal desde el caché en memoria (mucho más rápido)
        df_senal_cacheado = signal_data_cache[ruta_completa_csv]
        senal = df_senal_cacheado[id_sag].to_numpy()

    except KeyError:
        # print(f"ADVERTENCIA: No se encontró el archivo para el tipo '{tipo_evento}' o la señal '{id_sag}'. Saltando.")
        continue
    except Exception as e:
        print(f"Error procesando la señal {id_sag}: {e}")
        continue

    ground_truth = (fila['inicio_muestra'], fila['fin_muestra'])

    for nombre_alg, func_alg in algoritmos:
        start_time = time.perf_counter()
        intervalos_detectados = func_alg(senal)
        end_time = time.perf_counter()

        mejor_intervalo, mejor_iou = encontrar_mejor_coincidencia(ground_truth, intervalos_detectados)
        
        umbral_iou_tp = 0.5
        tp = 1 if mejor_iou >= umbral_iou_tp else 0
        fn = 1 if tp == 0 else 0
        fp = len(intervalos_detectados) - tp

        resultados.append({
            'ID_Sag': id_sag, 'Tipo_Evento': tipo_evento, 'Algoritmo': nombre_alg,
            'Tiempo_ms': (end_time - start_time) * 1000, 'IoU': mejor_iou,
            'TP': tp, 'FN': fn, 'FP': fp
        })

# --- ANÁLISIS Y GUARDADO DE RESULTADOS ---

resultados_df = pd.DataFrame(resultados)
ruta_resultados = os.path.join(BASE_DIR, 'resultados_evaluacion.csv')
resultados_df.to_csv(ruta_resultados, index=False)
print(f"\n✅ ¡Evaluación completada! Resultados guardados en '{ruta_resultados}'")

summary = resultados_df.groupby('Algoritmo').agg(
    Tiempo_Promedio_ms=('Tiempo_ms', 'mean'),
    IoU_Promedio=('IoU', 'mean'),
    Total_TP=('TP', 'sum'),
    Total_FN=('FN', 'sum'),
    Total_FP=('FP', 'sum')
).reset_index()

summary['Precision'] = summary['Total_TP'] / (summary['Total_TP'] + summary['Total_FP'])
summary['Sensibilidad'] = summary['Total_TP'] / (summary['Total_TP'] + summary['Total_FN'])
summary['F1_Score'] = 2 * (summary['Precision'] * summary['Sensibilidad']) / (summary['Precision'] + summary['Sensibilidad'])
summary = summary.fillna(0)

print("\n--- 📊 RESUMEN DE RENDIMIENTO POR ALGORITMO ---")
print(summary.to_string(index=False, float_format="%.4f"))

Cargando y consolidando metadatos...
  - Cargado: sags_metadata.csv (4100 filas)
  - Cargado: metadata_consolidado.csv (4100 filas)
  - Cargado: metadata_multiples.csv (4100 filas)
  - Cargado: metadata_notches.csv (4100 filas)
  - Cargado: metadata_transitorios.csv (4100 filas)

Total de señales a evaluar: 20500

Iniciando evaluación con caché de datos optimizado...


Evaluando señales: 100%|██████████| 20500/20500 [02:24<00:00, 141.78it/s]



✅ ¡Evaluación completada! Resultados guardados en '..\Generacion_Señales\resultados_evaluacion.csv'

--- 📊 RESUMEN DE RENDIMIENTO POR ALGORITMO ---
Algoritmo  Tiempo_Promedio_ms  IoU_Promedio  Total_TP  Total_FN  Total_FP  Precision  Sensibilidad  F1_Score
   Fusion              2.8546        0.8397     17981      2519      1338     0.9307        0.8771    0.9031
  Hilbert              0.7689        0.9192     19700       800     88162     0.1826        0.9610    0.3069
      RMS              1.2721        0.8427     18967      1533       779     0.9605        0.9252    0.9426
      SWT              0.8613        0.0625       263     20237     82291     0.0032        0.0128    0.0051


Cargando y consolidando metadatos...

Usando un subconjunto aleatorio de 500 señales para la optimización...
Cargando las señales del subconjunto...


Cargando datos: 100%|██████████| 500/500 [00:20<00:00, 23.91it/s]


Cargando las señales del subconjunto...


Cargando datos: 100%|██████████| 500/500 [00:00<00:00, 38326.55it/s]



Pre-calculando features base (RMS, Hilbert, SWT)... Este paso puede tardar un poco.


Pre-cálculo: 100%|██████████| 1000/1000 [03:38<00:00,  4.57it/s]



Iniciando Grid Search rápido con 108 combinaciones...


Probando combinaciones: 100%|██████████| 108/108 [02:13<00:00,  1.24s/it]



--- 🏆 MEJOR CONFIGURACIÓN ENCONTRADA (ordenada por F1-Score) ---
  - w_rms: 0.6000
  - w_h: 0.3000
  - w_w: 0.1000
  - thr_on: 0.4000
  - thr_off: 0.3000
  - F1: 0.7081
  - Precision: 0.7166
  - Recall: 0.6998
  - IoU: 0.5882

--- TOP 5 MEJORES COMBINACIONES ---
 w_rms    w_h    w_w  thr_on  thr_off     F1  Precision  Recall    IoU
0.6000 0.3000 0.1000  0.4000   0.3000 0.7081     0.7166  0.6998 0.5882
0.6000 0.3000 0.1000  0.4000   0.2500 0.7081     0.7164  0.6999 0.5882
0.6000 0.3000 0.0500  0.3500   0.2500 0.7080     0.7165  0.6997 0.5882
0.6000 0.3000 0.0500  0.3500   0.3000 0.7080     0.7166  0.6997 0.5881
0.6364 0.2727 0.0909  0.3500   0.2500 0.7077     0.7158  0.6998 0.5882
